<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Matías Godoy
- Nombre de alumno 2: Delaney Tello


Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Link de repositorio de GitHub:** [Repositorio](https://github.com/MDS7202-GPT-6/GPT-6)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [1]:
!pip install -qq xgboost optuna

# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv("sales.csv")

df.head()

,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
0,0,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,glass,500ml,0.96,13280
1,1,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,plastic,1.5lt,2.86,6727
2,2,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,can,330ml,0.87,9848
3,3,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,glass,500ml,1.00,20050
4,4,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,can,330ml,0.39,25696


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [40]:
from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
set_config(transform_output="pandas")

y = df["quantity"]
X = df.drop(columns=["quantity"])

# Split train, val, test (70%, 20%, 10%)
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.1, random_state=6)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.2222, random_state=6)



def extract_date_features(df):
    df = df.copy()
    # Convertir la fecha
    df["date"] = pd.to_datetime(df["date"],format="%d/%m/%Y", errors="coerce")
    df["day"] = df["date"].dt.day.astype("category")
    df["month"] = df["date"].dt.month.astype("category")
    df["year"] = df["date"].dt.year.astype("category")
    df = df.drop(columns=["date"])
    return df

date_transformer = FunctionTransformer(extract_date_features)

cat_features = ["city", "shop", "brand", "container", "capacity", "day", "month", "year"]
num_features = ["lat", "long", "pop", "price"]


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore",sparse_output=False), cat_features)
    ],
    remainder="drop"
).set_output(transform="pandas")






In [41]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

baseline_pipeline = Pipeline(steps=[
    ("date", date_transformer),
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])

baseline_pipeline.fit(X_train, y_train)
y_val_pred = baseline_pipeline.predict(X_val)
mae_dummy = mean_absolute_error(y_val, y_val_pred)

print(f"MAE DummyRegressor (validación): {mae_dummy:.2f}")

MAE DummyRegressor (validación): 13371.93


In [42]:
from xgboost import XGBRegressor
xgb_pipeline = Pipeline(steps=[
    ("date", date_transformer),
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(random_state=42))
])

xgb_pipeline.fit(X_train, y_train)
y_val_pred_xgb = xgb_pipeline.predict(X_val)
mae_xgb = mean_absolute_error(y_val, y_val_pred_xgb)
print(f"MAE XGBRegressor dummy (validación): {mae_dummy:.2f}")

MAE XGBRegressor dummy (validación): 13371.93


Podemos apreciar que el modelo 'DummyRegresor' versus 'XGBRegressor', obtienen exactamente el mismo MAE en el conjunto de validación.

In [43]:
import pickle
with open("baseline_dummy.pkl", "wb") as f:
    pickle.dump(baseline_pipeline, f)

with open("baseline_xgb.pkl", "wb") as f:
    pickle.dump(xgb_pipeline, f)

## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [44]:
# Pipeline temporal con Dummy (o XGB sin entrenar)
tmp_pipeline = Pipeline(steps=[
    ("date", date_transformer),
    ("preprocessor", preprocessor),
    ("model", DummyRegressor())
])

# Ajustamos para que cree todas las features
tmp_pipeline.fit(X_train, y_train)

# Nombres de features después del preprocessor
feature_names = tmp_pipeline.named_steps["preprocessor"].get_feature_names_out()
print(feature_names)

['num__lat' 'num__long' 'num__pop' 'num__price' 'cat__city_Athens'
 'cat__city_Irakleion' 'cat__city_Larisa' 'cat__city_Patra'
 'cat__city_Thessaloniki' 'cat__shop_shop_1' 'cat__shop_shop_2'
 'cat__shop_shop_3' 'cat__shop_shop_4' 'cat__shop_shop_5'
 'cat__shop_shop_6' 'cat__brand_adult-cola' 'cat__brand_gazoza'
 'cat__brand_kinder-cola' 'cat__brand_lemon-boost'
 'cat__brand_orange-power' 'cat__container_can' 'cat__container_glass'
 'cat__container_plastic' 'cat__capacity_1.5lt' 'cat__capacity_330ml'
 'cat__capacity_500ml' 'cat__day_nan' 'cat__month_nan' 'cat__year_nan']


In [45]:
# 1. Crear vector de restricciones
constraints = [0] * len(feature_names)
idx_price = list(feature_names).index("num__price")
constraints[idx_price] = -1   # Forzar relación inversa precio → cantidad

print("Restricciones aplicadas:", constraints)

# 2. Pipeline con XGB y constraint
xgb_mono_pipeline = Pipeline(steps=[
    ("date", date_transformer),
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        random_state=42,
        monotone_constraints={"num__price": -1}  # restringimos SOLO el precio
    ))
])
# 3. Entrenar
xgb_mono_pipeline.fit(X_train, y_train)

# 4. Evaluar en validación
y_val_pred_mono = xgb_mono_pipeline.predict(X_val)
mae_xgb_mono = mean_absolute_error(y_val, y_val_pred_mono)

print(f"MAE XGB con restricción monotónica: {mae_xgb_mono:.2f}")



Restricciones aplicadas: [0, 0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
MAE XGB con restricción monotónica: 7250.15


Se puede apreciar que el modelo con la restricción monotónica, obtiene un MAE mucho más bajo que el valor inicial, ya que baja desde 13400 aproximadamente a 7250 aproximadamente.

In [46]:
# 5. Guardar modelo
with open("baseline_xgb_monotone.pkl", "wb") as f:
    pickle.dump(xgb_mono_pipeline, f)

## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [47]:
import optuna
from optuna.samplers import TPESampler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import pickle

SEED = 6
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    # Hiperparámetros a optimizar
    lr = trial.suggest_float("learning_rate", 0.001, 0.1, log=True)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 1.0)
    ohe_min_freq = trial.suggest_float("ohe_min_frequency", 1e-6, 0.999)

    # Preprocesador con min_frequency dinámico
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_features),
            ("cat", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                min_frequency=ohe_min_freq
            ), cat_features)
        ],
        remainder="drop"
    ).set_output(transform="pandas")

    # Modelo con restricción monotónica en price
    model = XGBRegressor(
        random_state=SEED,
        tree_method="hist",
        learning_rate=lr,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        monotone_constraints={"num__price": -1}
    )

    # Pipeline completo
    pipe = Pipeline(steps=[
        ("date", FunctionTransformer(extract_date_features)),
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Entrenar y evaluar en validación
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)
    mae = mean_absolute_error(y_val, preds)

    # Guardar el pipeline entrenado en el trial
    trial.set_user_attr("pipeline", pipe)
    return mae


In [48]:
sampler = TPESampler(seed=SEED)
study = optuna.create_study(direction="minimize", sampler=sampler)

# 5 minutos de búsqueda
study.optimize(objective, timeout=300)

best_trial = study.best_trial
print("Trials ejecutados:", len(study.trials))
print("Mejor MAE (validación):", best_trial.value)
print("Mejores hiperparámetros:")
for k, v in best_trial.params.items():
    print(f"  - {k}: {v}")

# Recuperar mejor pipeline y guardarlo
best_pipeline = best_trial.user_attrs["pipeline"]
with open("best_xgb_optuna.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)
print("Modelo guardado en best_xgb_optuna.pkl")

Trials ejecutados: 637
Mejor MAE (validación): 6756.379869385584
Mejores hiperparámetros:
  - learning_rate: 0.04750815781979247
  - n_estimators: 611
  - max_depth: 3
  - max_leaves: 82
  - min_child_weight: 5
  - reg_alpha: 0.37782580808302435
  - reg_lambda: 0.9367791919733419
  - ohe_min_frequency: 0.1873661075828515
Modelo guardado en best_xgb_optuna.pkl


## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [22]:
!pip install "optuna-integration[xgboost]"

  Using cached optuna_integration-4.5.0-py3-none-any.whl.metadata (12 kB)


In [27]:
!pip install --upgrade xgboost

In [49]:
import optuna
from optuna.samplers import TPESampler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import pickle

SEED = 6
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # ---- Espacios de búsqueda ----
    lr = trial.suggest_float("learning_rate", 0.001, 0.1, log=True)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.0, 1.0)
    ohe_min_freq = trial.suggest_float("ohe_min_frequency", 1e-6, 0.999)

    # Asegurar que las categóricas estén en formato category (para OneHotEncoder)
    X_train_clean = X_train.copy()
    X_val_clean = X_val.copy()
    for col in ["city", "shop", "brand", "container", "capacity"]:
        X_train_clean[col] = X_train_clean[col].astype("category")
        X_val_clean[col] = X_val_clean[col].astype("category")

    # Preprocesador dinámico
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_features),
            ("cat", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                min_frequency=ohe_min_freq
            ), cat_features)
        ],
        remainder="drop"
    ).set_output(transform="pandas")

    # Modelo XGB con monotonicidad en precio
    xgb = XGBRegressor(
        random_state=SEED,
        tree_method="hist",
        learning_rate=lr,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        monotone_constraints={"num__price": -1},
        eval_metric="mae"
    )

    # Pipeline completo
    pipe = Pipeline(steps=[
        ("date", FunctionTransformer(extract_date_features)),
        ("preprocessor", preprocessor),
        ("model", xgb)
    ])

    # 🚨 Entrenamos SIN eval_set (porque rompe con objetos)
    pipe.fit(X_train_clean, y_train)

    # Evaluamos en validación con pipeline ya entrenado
    preds = pipe.predict(X_val_clean)
    mae = mean_absolute_error(y_val, preds)

    # Reportamos y pruning manual
    trial.report(mae, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    trial.set_user_attr("pipeline", pipe)
    return mae

In [50]:

# ========== Optimización con Pruning manual ==========
sampler = TPESampler(seed=SEED)
study_pruning = optuna.create_study(direction="minimize", sampler=sampler, pruner=optuna.pruners.MedianPruner())

study_pruning.optimize(objective, timeout=300, show_progress_bar=True)

# Resultados
best_trial = study_pruning.best_trial
print("\n===== RESULTADOS PRUNNING (manual) =====")
print("Trials ejecutados:", len(study_pruning.trials))
print("Mejor MAE (validación):", best_trial.value)
print("Mejores hiperparámetros:")
for k, v in best_trial.params.items():
    print(f"  - {k}: {v}")

# Guardar el mejor modelo
best_pipeline = best_trial.user_attrs["pipeline"]
with open("best_xgb_optuna_pruning.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)
print("Modelo guardado en best_xgb_optuna_pruning.pkl")

   0%|          | 00:00/05:00


===== RESULTADOS PRUNNING (manual) =====
Trials ejecutados: 622
Mejor MAE (validación): 6756.379869385584
Mejores hiperparámetros:
  - learning_rate: 0.04750815781979247
  - n_estimators: 611
  - max_depth: 3
  - max_leaves: 82
  - min_child_weight: 5
  - reg_alpha: 0.37782580808302435
  - reg_lambda: 0.9367791919733419
  - ohe_min_frequency: 0.1873661075828515
Modelo guardado en best_xgb_optuna_pruning.pkl


## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [51]:
# Inserte su código acá
import optuna.visualization as vis

# 1. Historial de optimización
fig_hist = vis.plot_optimization_history(study)
fig_hist.show()

# 2. Gráfico de coordenadas paralelas
fig_parallel = vis.plot_parallel_coordinate(study)
fig_parallel.show()

# 3. Importancia de hiperparámetros
fig_importance = vis.plot_param_importances(study)
fig_importance.show()

In [52]:
# Inserte su código acá
import optuna.visualization as vis

# 1. Historial de optimización
fig_hist = vis.plot_optimization_history(study_pruning)
fig_hist.show()

# 2. Gráfico de coordenadas paralelas
fig_parallel = vis.plot_parallel_coordinate(study_pruning)
fig_parallel.show()

# 3. Importancia de hiperparámetros
fig_importance = vis.plot_param_importances(study_pruning)
fig_importance.show()

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [ ]:
# Inserte su código acá
import pandas as pd

results = pd.DataFrame({
    "Modelo": [
        "DummyRegressor (Baseline)",
        "XGBRegressor (Default)",
        "XGBRegressor (Monotonic Constraint)",
        "XGBRegressor (Optuna)",
        "XGBRegressor (Optuna + Prunning)"
    ],
    "MAE_Validación": [
        mae_dummy,
        mae_xgb,
        mae_xgb_mono,
        study.best_trial.value,
        study_pruning.best_trial.value  
    ]
})

print(results)

                                Modelo  MAE_Validación
0            DummyRegressor (Baseline)    13371.934823
1               XGBRegressor (Default)     7216.021994
2  XGBRegressor (Monotonic Constraint)     7250.148252
3                XGBRegressor (Optuna)     6756.379869
4     XGBRegressor (Optuna + Prunning)     6756.379869


# Conclusión
Exito!
<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>